In [1]:
# Bootstrap : réutilise l'aplatissement d'analyse.ipynb comme source de vérité
# 
# Ce notebook a son propre kernel : `_snake`, `schema`, `bases` n'existent pas tant qu'on ne les importe pas. 
# Plutôt que de dupliquer le parseur XML (deux sources de vérité = dérive garantie), on exécute analyse.ipynb dans CE kernel.
# 
# Effet de bord : 
# 
# %run réexécute TOUT analyse.ipynb, y compris ses print_base/
#    print_schema (sortie verbeuse) et le parsing de /mnt/data/Murphy/src.
#    io.capture_output() avale cette sortie pour garder ce notebook lisible.
# 
# Après cette cellule sont définis : _snake, schema, bases, analyze, Node, ...
#    from IPython.utils import io\n\nwith io.capture_output(): 
#    avale la sortie verbeuse d'analyse.ipynb

In [38]:
%run analyse.ipynb
assert "schema" in dir() and "_snake" in dir() and "bases" in dir(), ("analyse.ipynb n'a pas défini schema/_snake/bases — vérifier son contenu")
print("bootstrap OK :", bases, "| schema pour", list(schema))

bases = bases
schema = schema
_snake = _snake

CAPP: 1 fichiers
CASS: 92 fichiers
CONSTIT: 2 fichiers
INCA: 1 fichiers
JADE: 256 fichiers
### CAPP  (1 documents)

— arbre: TEXTE_JURI_JUDI —
TEXTE_JURI_JUDI  ×1
├─ META  ×1
│  ├─ META_COMMUN  ×1
│  │  ├─ ID  ×1  «txt»
│  │  ├─ ANCIEN_ID  ×1
│  │  ├─ ORIGINE  ×1  «txt»
│  │  ├─ URL  ×1  «txt»
│  │  └─ NATURE  ×1  «txt»
│  └─ META_SPEC  ×1
│     ├─ META_JURI  ×1
│     │  ├─ TITRE  ×1  «txt»
│     │  ├─ DATE_DEC  ×1  «txt»
│     │  ├─ JURIDICTION  ×1  «txt»
│     │  ├─ NUMERO  ×1  «txt»
│     │  └─ SOLUTION  ×1
│     └─ META_JURI_JUDI  ×1
│        ├─ NUMEROS_AFFAIRES  ×1
│        │  └─ NUMERO_AFFAIRE  ×1  «txt»
│        ├─ PUBLI_BULL  ×1
│        │  └─ @publie
│        ├─ FORMATION  ×1  «txt»
│        ├─ FORM_DEC_ATT  ×1  «txt»
│        ├─ DATE_DEC_ATT  ×1  «txt»
│        ├─ SIEGE_APPEL  ×1  «txt»
│        ├─ JURI_PREM  ×1
│        ├─ LIEU_PREM  ×1
│        ├─ DEMANDEUR  ×1
│        ├─ DEFENDEUR  ×1
│        ├─ PRESIDENT  ×1
│        ├─ AVOCAT_GL  ×1
│        ├─ AVOCATS  ×1
│        └─ 

## 1. Extraction des champs — nom = chemin complet

On applique la convention de `notes.txt` : le nom d'un champ est le **chemin complet**
des tags ancêtres en snake_case, joint par `_`, suivi du segment final —

- pour un **attribut** : le nom de l'attribut (`<TAG a="A">` → `..._tag_a`) ;
- pour le **texte** d'un tag : le suffixe littéral `text_` (`<TAG>Text</TAG>` → `..._tag_text_`).

L'underscore final de `text_` évite toute collision avec une future clé nommée « text ».
Aucune désambiguïsation n'est nécessaire : le chemin complet est injectif par construction
(prouvé côté MongoDB dans `analyse.ipynb`).

In [3]:
from collections import namedtuple

# Un champ = une paire clé→valeur candidate au dict `metadata`.
#   segments : chemin XML en snake_case, du plus haut au segment final
#              (nom d'attribut, ou "text_" pour du contenu textuel)
#   key      : "_".join(segments) = le nom final, chemin complet (injectif par construction)
Field = namedtuple("Field", "segments key")


def _walk_fields(node, ancestors):
    """Émet un Field par attribut et par feuille textuelle (mêmes règles de parcours
    que analyse.ipynb), en nommant chaque champ par son chemin complet."""
    path = ancestors + [_snake(node["name"])]

    # attributs : segment final = nom d'attribut → <chemin>_<attr>
    for a in node["attribs"]:
        segs = path + [_snake(a["name"])]
        yield Field(segs, "_".join(segs))

    # feuille textuelle : segment final = "text_" → <chemin>_text_
    if node["text"]["status"] != "never":
        segs = path + ["text_"]
        yield Field(segs, "_".join(segs))

    for c in node["children"]:
        yield from _walk_fields(c, path)


def extract_raw_fields(base_result):
    """-> liste de Field pour une base (sortie de analyze()[base]). Dédupliqué par key."""
    seen, out = set(), []
    for tree in base_result["trees"]:
        for f in _walk_fields(tree, []):
            if f.key in seen:
                continue
            seen.add(f.key)
            out.append(f)
    return out

## 2. Résultat par base — les champs finaux

Pour chaque `document_type`, on liste les noms de champs produits (chemin complet, triés).
C'est exactement ce qui atterrirait dans le dict `metadata` à l'ingestion.

In [4]:
def print_fields(base_result, base_name=""):
    fields = extract_raw_fields(base_result)
    keys = sorted(f.key for f in fields)

    print(f"### {base_name} \n")
    for key in keys:
        print(f"  {key}")
    print()

for base in bases:
    print_fields(schema[base], base)

### CAPP 

  texte_juri_judi_meta_meta_commun_id_text_
  texte_juri_judi_meta_meta_commun_nature_text_
  texte_juri_judi_meta_meta_commun_origine_text_
  texte_juri_judi_meta_meta_commun_url_text_
  texte_juri_judi_meta_meta_spec_meta_juri_date_dec_text_
  texte_juri_judi_meta_meta_spec_meta_juri_judi_date_dec_att_text_
  texte_juri_judi_meta_meta_spec_meta_juri_judi_form_dec_att_text_
  texte_juri_judi_meta_meta_spec_meta_juri_judi_formation_text_
  texte_juri_judi_meta_meta_spec_meta_juri_judi_numeros_affaires_numero_affaire_text_
  texte_juri_judi_meta_meta_spec_meta_juri_judi_publi_bull_publie
  texte_juri_judi_meta_meta_spec_meta_juri_judi_siege_appel_text_
  texte_juri_judi_meta_meta_spec_meta_juri_juridiction_text_
  texte_juri_judi_meta_meta_spec_meta_juri_numero_text_
  texte_juri_judi_meta_meta_spec_meta_juri_titre_text_
  texte_juri_judi_texte_bloc_textuel_contenu_text_

### CASS 

  texte_juri_judi_liens_lien_cidtexte
  texte_juri_judi_liens_lien_datesignatexte
  texte_juri

## 3. Démo valeur — concaténation du texte des balises transparentes

Les notebooks ne font sinon que de la **statistique de schéma** ; ici on lit un vrai
fichier XML pour **matérialiser la valeur** d'un champ textuel avec la transparence
appliquée.

Règle : `<p>` sépare par `\n\n`, `<br/>` par `\n` (cf. `TRANSPARENT_TAGS` d'`analyse.ipynb`),
les fragments de texte sont `strip()`és puis joints par le séparateur entouré d'espaces.

In [40]:
import xml.etree.ElementTree as ET
# TRANSPARENT_TAGS, strip_ns viennent d'analyse.ipynb (bootstrap %run).


def concat_text(elem, strip=True):
    """Valeur textuelle réelle de `elem` avec transparence des balises produit.

    Parcourt le contenu mixte dans l'ordre du document ; chaque balise transparente
    (p, br, …) insère son séparateur (TRANSPARENT_TAGS) au lieu d'apparaître. Les
    fragments de texte sont strip()és, les séparateurs entourés d'espaces, le tout joint.
    Ne descend PAS dans les enfants non transparents (leur texte est un autre champ)."""
    tokens = []  # ("txt", s) | ("sep", s)

    def emit_text(s):
        if s and s.strip():
            tokens.append(("txt", s.strip()))

    def walk(e):
        emit_text(e.text)
        for c in e:
            tag = strip_ns(c.tag) if strip else c.tag
            if tag in TRANSPARENT_TAGS:
                sep = TRANSPARENT_TAGS[tag]
                tokens.append(("sep", sep))   # ouverture du bloc / saut
                walk(c)                        # contenu du transparent remonte ici
                tokens.append(("sep", sep))   # fermeture du bloc
            emit_text(c.tail)                  # texte APRÈS la balise (contenu mixte)

    walk(elem)

    # Assemble : entre deux fragments texte, on garde le séparateur le plus fort
    # rencontré, entouré d'espaces. Les séparateurs en tête/queue sont ignorés.
    out, pending = [], None
    for kind, s in tokens:
        if kind == "txt":
            if out:
                out.append(f" {pending or ''} " if pending else " ")
            out.append(s)
            pending = None
        else:  # sep : mémorise le plus fort (\n\n > \n) jusqu'au prochain texte
            if pending is None or len(s) > len(pending):
                pending = s
    return "".join(out)


def extract_metadata(path):
    """Dict `metadata` réel d'un document XML : {clé chemin-complet: valeur}.

    Applique EXACTEMENT les mêmes règles que le reste du notebook :
      - transparence p/br (via concat_text, qui déplie leur texte) ;
      - nommage chemin-complet snake_case + `_text_` (texte) / nom d'attribut ;
    Une clé rencontrée plusieurs fois → valeur = liste ; sinon valeur scalaire."""
    root = ET.parse(path).getroot()
    out = {}

    def add(key, value):
        if key in out:
            if not isinstance(out[key], list):
                out[key] = [out[key]]
            out[key].append(value)
        else:
            out[key] = value

    def walk(elem, ancestors):
        path_segs = ancestors + [strip_ns(elem.tag).lower()]

        # attributs → <chemin>_<attr>
        for name, val in elem.attrib.items():
            add("_".join(path_segs + [strip_ns(name).lower()]), val)

        # texte propre (transparents dépliés) → <chemin>_text_
        txt = concat_text(elem)
        if txt:
            add("_".join(path_segs + ["text_"]), txt)

        # enfants NON transparents (les p/br ont déjà été absorbés par concat_text)
        for c in elem:
            if strip_ns(c.tag) not in TRANSPARENT_TAGS:
                walk(c, path_segs)

    walk(root, [])
    return out



for base in bases:
    print(f"\n### {base}\n")
    # Démo : dict clé→valeur d'un vrai fichier CAPP (contient des <p>/<br>).
    path = glob(f"/mnt/data/Murphy/src/{base}/**/*.xml", recursive=True)[0]
    print(path, "\n")
    for key, value in extract_metadata(path).items():
        print(f"{key} = {value!r}")


### CAPP

/mnt/data/Murphy/src/CAPP/CAPP_20260612-215918/20260612-215918/juri/capp/global/JURI/TEXT/00/00/19/33/38/JURITEXT000019333891.xml 

texte_juri_judi_meta_meta_commun_id_text_ = 'JURITEXT000019333891'
texte_juri_judi_meta_meta_commun_origine_text_ = 'JURI'
texte_juri_judi_meta_meta_commun_url_text_ = 'texte/juri/judi/JURI/TEXT/00/00/19/33/38/JURITEXT000019333891.xml'
texte_juri_judi_meta_meta_commun_nature_text_ = 'ARRET'
texte_juri_judi_meta_meta_spec_meta_juri_titre_text_ = "Cour d'appel d'Aix-en-Provence, Chambre civile 1, 6 mars 2008, 07/01993"
texte_juri_judi_meta_meta_spec_meta_juri_date_dec_text_ = '2008-03-06'
texte_juri_judi_meta_meta_spec_meta_juri_juridiction_text_ = "Cour d'appel d'Aix-en-Provence"
texte_juri_judi_meta_meta_spec_meta_juri_numero_text_ = '225'
texte_juri_judi_meta_meta_spec_meta_juri_judi_numeros_affaires_numero_affaire_text_ = '07/01993'
texte_juri_judi_meta_meta_spec_meta_juri_judi_publi_bull_publie = 'non'
texte_juri_judi_meta_meta_spec_meta_juri